# MedRAX on Colab

Runs the `edv-forced-validation` branch on a GPU. Colab is Linux + CUDA, which is what
MedRAX assumes, so tools that cannot run on Apple Silicon work here.

**Do the measurement first, the UI second.** The reliability table is what the evidence
weighting depends on, and one finding is still missing from it.

**Runtime → Change runtime type → GPU** before you start.


## 1. Check the machine

In [ ]:
!nvidia-smi || echo "NO GPU -- Runtime > Change runtime type > GPU, then rerun"
!python --version
# Colab's kernel is Python 3.13; this project needs 3.11. The next cell installs a
# separate 3.11 via uv rather than fighting the kernel, so 3.13 here is expected.

## 2. Clone and install

Colab ships Python 3.13. This project pins `numpy<2` and `tokenizers 0.19.1`, and
neither publishes cp313 wheels, so a plain `pip install -e .` tries to build them from
source and fails.

`uv` fetches its own CPython 3.11 alongside Colab's kernel. Everything afterwards runs
through `.venv/bin/python` rather than `!python`. The kernel's own packages are never
touched, so **no runtime restart is needed** either.

In [ ]:
!pip install -q uv
!git clone -q https://github.com/vaniinfo/MedRAX.git /content/MedRAX
%cd /content/MedRAX
!git checkout -q edv-forced-validation

# A managed CPython 3.11, independent of Colab's 3.13 kernel
!uv venv --python 3.11 .venv
!uv pip install --python .venv/bin/python -e .

!.venv/bin/python --version
!.venv/bin/python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 3. After the restart: weights, key, environment

Weights are ~14 GB (~28 GB with LLaVA-Med). Keep them on Drive or you re-download
every session — free Colab disconnects after about 90 minutes idle.

In [ ]:
%cd /content/MedRAX
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ["MEDRAX_MODEL_DIR"] = "/content/drive/MyDrive/medrax-weights"
os.makedirs(os.environ["MEDRAX_MODEL_DIR"], exist_ok=True)

# Secrets tab (key icon, left sidebar). Never paste the key into a cell.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["MEDRAX_SHARE"] = "1"      # Colab has no reachable localhost

# Env vars set here are inherited by the ! cells below.
!.venv/bin/python -c "import torch; print('cuda:', torch.cuda.get_device_name(0))"

## 4. Verify the vision path BEFORE trusting any output

CheXagent can load, generate fluent clinical text, and ignore the image entirely —
returning the same answer for a real X-ray and a blank one, with no error. That
happened for hours on the Mac before it was noticed. This asserts that answers
actually depend on the image. **Exit code 0 = healthy.**

In [ ]:
!.venv/bin/python scripts/verify_vqa_vision.py

## 5. The measurement — this is the point of running on Colab

Downloads a labelled evaluation set from NIH Open-i (same collection as the Kaggle
Indiana University dataset, but the API also returns the curated MeSH labels used as
ground truth).

**Known gap: pneumothorax.** Its query timed out during the Mac run, leaving only 3
positives, so it has no measured row and falls back to an ASSUMED 0.5 threshold —
and it is the finding this whole branch was built around. Check the class balance
below and re-run this cell if pneumothorax is still thin.

In [ ]:
!.venv/bin/python scripts/fetch_openi_eval_set.py

In [ ]:
import csv
rows = list(csv.DictReader(open('data/indiana_eval/GROUND_TRUTH.csv')))
print(f"images: {len(rows)}\n")
for finding in ["cardiomegaly", "pleural effusion", "pneumothorax",
                "pulmonary edema", "atelectasis", "nodule"]:
    pos = sum(1 for r in rows if finding in r["problems"].lower())
    flag = "  <-- too few to measure" if pos < 20 else ""
    print(f"  {finding:18s} positives={pos:3d}  negatives={len(rows)-pos:3d}{flag}")

### Run the tools

One classifier call and one report-generator call per image, plus one CheXagent call
per finding. Checkpoints every 10 images and resumes, so a disconnect costs little.
No API calls — local inference only.

In [ ]:
!.venv/bin/python scripts/measure_reliability.py

### The table

`AUC` is the number that matters: 0.5 is chance. `best thr` is where balanced accuracy
peaks — on the Mac run it ranged from 0.20 to 0.70, which is why a fixed 0.5 cutoff
or a band centred on 0.5 discards real signal.

In [ ]:
import json

R = json.load(open("reliability.json"))
findings = list(R[0]["findings"].keys())

def auc(pairs):
    pos = [p for p, t in pairs if t]; neg = [p for p, t in pairs if not t]
    if not pos or not neg: return float("nan")
    return sum((a > b) + 0.5 * (a == b) for a in pos for b in neg) / (len(pos) * len(neg))

def at(pairs, thr):
    tp = sum(1 for p, t in pairs if t and p >= thr); fn = sum(1 for p, t in pairs if t and p < thr)
    fp = sum(1 for p, t in pairs if not t and p >= thr); tn = sum(1 for p, t in pairs if not t and p < thr)
    sens = tp / (tp + fn) if tp + fn else float("nan")
    spec = tn / (tn + fp) if tn + fp else float("nan")
    return sens, spec, (sens + spec) / 2

print(f"images: {len(R)}\n")
print(f"{'finding':18s} {'n+':>4s} {'tool':12s} {'AUC':>6s} {'best thr':>9s} {'balanced':>9s}")
for f in findings:
    npos = sum(1 for r in R if r["findings"][f]["truth"])
    for tool, key in [("CheXagent", "chexagent_p"), ("classifier", "classifier_p")]:
        pairs = [(r["findings"][f][key], r["findings"][f]["truth"])
                 for r in R if r["findings"][f].get(key) is not None]
        if not pairs: continue
        best = max(((t / 100, *at(pairs, t / 100)) for t in range(5, 100, 5)), key=lambda x: x[3])
        note = "   (few positives - noisy)" if npos < 30 else ""
        print(f"{f:18s} {npos:4d} {tool:12s} {auc(pairs):6.3f} {best[0]:9.2f} {best[3]:9.1%}{note}")
print("\nCopy any improved rows into RELIABILITY in medrax/agent/validator.py.")

## 6. Optional: the chat UI

`MEDRAX_SHARE=1` gives a public `gradio.live` link, since Colab has no reachable
localhost. **That link is public while the cell runs** — do not put patient data
through it. Note also that every uploaded image is sent to the OpenAI API.

In [ ]:
!.venv/bin/python main.py

## 7. Optional: enable LLaVA-Med

Disabled on Apple Silicon because of hardcoded `.cuda()` calls and bitsandbytes;
both work here. Uncomment `"LlavaMedTool"` in `main.py`.

VRAM on a free T4 (16 GB): CheXagent ~6 GB + LLaVA-Med 8-bit ~8 GB + others ~2 GB
**will OOM.** Use `load_in_4bit=True` (~5 GB) or skip it unless you have an L4/A100.

It returns plain text with no probability, so the validator treats it like the report
generator — the weakest tool measured, at 26–78% precision. **Measure it before you
trust it**, or you have added a second unmeasured text tool of exactly the kind that
caused the failures this branch exists to fix.

In [ ]:
!sed -i 's|# "LlavaMedTool",|"LlavaMedTool",|' main.py
!sed -i 's|load_in_8bit=True|load_in_4bit=True|' main.py    # T4: 4-bit or OOM
!grep -n "LlavaMedTool" main.py